In [5]:
import pandas as pd
import random
from typing import List, Tuple
import re

In [6]:
def generate_title_word_swaps(df: pd.DataFrame, 
                            num_augmentations: int,
                            title_left_col: str = 'title_left',
                            title_right_col: str = 'title_right',
                            pair_id_col: str = 'pair_id') -> pd.DataFrame:
    """
    Generate random word swaps between left and right titles for data augmentation.
    
    Args:
        df: Input DataFrame containing the dataset
        num_augmentations: Number of augmented versions to generate per example
        title_left_col: Name of the column containing left titles
        title_right_col: Name of the column containing right titles
        pair_id_col: Name of the column containing pair IDs
    
    Returns:
        DataFrame containing original and augmented examples
    """
    augmented_rows = []
    print(f"Starting title word swap augmentation with {num_augmentations} augmentations per example...")

    for index, row in df.iterrows():
        # Get the original titles
        original_title_left = row[title_left_col]
        original_title_right = row[title_right_col]
        
        if pd.isna(original_title_left) or pd.isna(original_title_right):
            continue

        # Split titles into words and clean
        words_left = [w.strip() for w in re.findall(r'\b\w+\b', original_title_left.lower())]
        words_right = [w.strip() for w in re.findall(r'\b\w+\b', original_title_right.lower())]
        
        if not words_left or not words_right:
            continue

        # Generate n augmented versions
        for aug_num in range(num_augmentations):
            # Randomly determine number of words to swap (1 to min(1, min(len(words_left), len(words_right))))
            max_swaps = min(len(words_left), len(words_right))
            num_words_to_swap = random.randint(1, max_swaps)
            
            # Randomly select words to swap from both titles
            words_to_swap_left = random.sample(range(len(words_left)), num_words_to_swap)
            words_to_swap_right = random.sample(range(len(words_right)), num_words_to_swap)
            
            # Create a copy of the original row
            new_row = row.copy()
            
            # Create new titles with swapped words
            new_words_left = words_left.copy()
            new_words_right = words_right.copy()
            
            # Swap words between titles
            for i in range(num_words_to_swap):
                idx_left = words_to_swap_left[i]
                idx_right = words_to_swap_right[i]
                new_words_left[idx_left], new_words_right[idx_right] = new_words_right[idx_right], new_words_left[idx_left]
            
            # Reconstruct the titles while preserving original case and punctuation
            new_title_left = original_title_left
            new_title_right = original_title_right
            
            # Update left title
            for i, (old_word, new_word) in enumerate(zip(words_left, new_words_left)):
                if old_word != new_word:
                    pattern = re.compile(r'\b' + re.escape(old_word) + r'\b', re.IGNORECASE)
                    new_title_left = pattern.sub(new_word, new_title_left)
            
            # Update right title
            for i, (old_word, new_word) in enumerate(zip(words_right, new_words_right)):
                if old_word != new_word:
                    pattern = re.compile(r'\b' + re.escape(old_word) + r'\b', re.IGNORECASE)
                    new_title_right = pattern.sub(new_word, new_title_right)
            
            # Update the row with new titles
            new_row[title_left_col] = new_title_left
            new_row[title_right_col] = new_title_right
            
            # Update pair_id to reflect the augmentation
            swapped_indices = '_'.join([f"L{idx}" for idx in sorted(words_to_swap_left)] + 
                                     [f"R{idx}" for idx in sorted(words_to_swap_right)])
            new_row[pair_id_col] = f"{row[pair_id_col]}_wordswap_{swapped_indices}_aug{aug_num}"
            
            augmented_rows.append(new_row)
            
            if index == 0:
                print(f"  Row {index}: Created augmentation {aug_num + 1} with {num_words_to_swap} word swaps")

    print(f"Augmentation complete. Total augmented rows: {len(augmented_rows)}")
    return pd.DataFrame(augmented_rows)

In [18]:
# load train_set
train_set = pd.read_pickle("../../data/wdc/train_small/preprocessed_wdcproducts80cc20rnd000un_train_small.pkl.gz", compression="gzip")

# only get the positive examples
matches = train_set[train_set['label'] == 1]

# calculate how many times we need to augment the dataset to get a 1:1 ratio
augmentation_ratio = (len(train_set) - 2*len(matches)) / len(matches)
print(f"Augmentation ratio: {augmentation_ratio}")

augmentation_dataset = generate_title_word_swaps(matches, num_augmentations=int(augmentation_ratio))

# merge the augmentation dataset with the matches
train_set = pd.concat([train_set, augmentation_dataset])

# shuffle the dataset
train_set = train_set.sample(frac=1).reset_index(drop=True)

# get label distribution
label_distribution = train_set['label'].value_counts()
print(label_distribution)

# save the dataset
train_set.to_pickle("../../data/wdc/train_small/augmentation/preprocessed_wdcproducts80cc20rnd000un_train_small_random_word_swap_1_1.pkl.gz", compression="gzip")


Augmentation ratio: 3.0
Starting title word swap augmentation with 3 augmentations per example...
  Row 0: Created augmentation 1 with 8 word swaps
  Row 0: Created augmentation 2 with 6 word swaps
  Row 0: Created augmentation 3 with 7 word swaps
Augmentation complete. Total augmented rows: 1500
label
1    2000
0    2000
Name: count, dtype: int64
